# 数据集转换为GLM格式

将 Hugging Face 上的医疗问答数据集转换为 ChatGLM 系列模型所要求的多轮对话 JSON 格式，并保存为本地 JSONL 文件，供后续微调训练使用。

## 一、导入依赖库与加载数据集

In [ ]:
from datasets import load_dataset  # 从 Hugging Face datasets 库导入 load_dataset 函数，用于在线加载数据集
import json                         # 导入 Python 内置 json 模块，用于序列化 Python 对象为 JSON 字符串

# load_dataset 返回类型：当指定 split 参数后返回 datasets.Dataset 对象（单个数据集切片）
# 参数说明：
#   第一个位置参数 : Hugging Face Hub 上的数据集仓库路径（字符串）
#   'zh'           : config_name（子集名称），此处选择中文医疗问答数据
#   split="train[:1%]" : 只加载训练集前 1% 的样本，用于快速验证流程；正式训练时改为 "train"
#   trust_remote_code=True : 允许执行数据集仓库中的自定义 Python 脚本（部分数据集必须开启）
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",  # Hub 数据集路径
    'zh',                                             # 中文子集
    split="train[:1%]",                               # 仅取前 1% 以加速调试
    trust_remote_code=True                            # 信任并运行仓库自定义加载代码
)
# dataset 类型 : datasets.Dataset
# dataset 包含的列（字段）: "Question"（str，医疗问题）、"Response"（str，参考回答）等

## 二、定义GLM多轮对话格式模板

In [ ]:
# ChatGLM 系列模型微调时，每条训练样本须符合以下多轮对话结构：
# {
#     "conversations": [
#         {"role": "user",      "content": "<用户输入>"},
#         {"role": "assistant", "content": "<模型回答>"}
#     ]
# }
# 此处先定义含占位符的模板字典，后续循环中再逐条替换具体内容
train_prompt_style = {
    "conversations": [                                  # 对话轮次列表，类型: list[dict]
        {"role": "user",      "content": "问题"},       # 第 0 轮：用户角色，content 为占位符
        {"role": "assistant", "content": "答案"}        # 第 1 轮：助手角色，content 为占位符
    ]
}
# train_prompt_style 类型 : dict
# 键 "conversations" 的值类型 : list，每个元素为含 "role"（str）和 "content"（str）的 dict

## 三、遍历数据集、转换格式并写入JSONL文件

In [ ]:
# open() 以写入模式（"w"）打开文件，encoding="utf-8" 确保中文字符不乱码
# with 语句会在代码块执行完毕后自动调用 f.close()，防止文件句柄泄漏
# f 类型 : _io.TextIOWrapper（文本文件对象）
with open("train_data.json", "w", encoding="utf-8") as f:

    # enumerate(dataset) 返回 (idx: int, item: dict) 的迭代器
    # idx  : 当前样本在数据集中的整数索引（从 0 开始）
    # item : 当前样本字典，包含 "Question"（str）、"Response"（str）等键
    for idx, item in enumerate(dataset):

        # dict.copy() 执行浅拷贝，生成模板的独立副本
        # 若直接赋值（data_entry = train_prompt_style），所有轮次共享同一对象，会导致内容覆盖
        # data_entry 类型 : dict，结构与 train_prompt_style 完全相同
        data_entry = train_prompt_style.copy()

        # 将第 0 轮（user）的 content 替换为当前样本的实际医疗问题文本
        # item["Question"] 类型 : str
        data_entry["conversations"][0]["content"] = item["Question"]

        # 将第 1 轮（assistant）的 content 替换为当前样本的实际参考回答文本
        # item["Response"] 类型 : str
        data_entry["conversations"][1]["content"] = item["Response"]

        # json.dumps(obj, ensure_ascii=False) 将 dict 序列化为 JSON 格式字符串
        #   ensure_ascii=False : 保留 UTF-8 中文字符原样输出，不转为 \uXXXX 转义
        # + "\n" : 每条 JSON 对象末尾加换行，形成 JSONL（JSON Lines）格式
        # JSONL 文件每行一条独立 JSON，读取时按行迭代即可，适合大规模数据集
        f.write(json.dumps(data_entry, ensure_ascii=False) + "\n")

# 所有样本写入完毕后打印提示信息
print("已生成 train_data.json 文件。")  # 控制台输出：已生成 train_data.json 文件。